* Notebook [riksdagen_64_wikidata.ipynb](https://github.com/salgo60/SCB-Wikidata/blob/main/notebook/riksdagen_64_wikidata.ipynb) 
* [#64](https://github.com/salgo60/SCB-Wikidata/issues/64)
* result   
   * [html](https://salgo60.github.io/SCB-Wikidata/notebook/resultsRiksdagenWD/links_RiksdagenWikidata_v1_2026_02_08.html) 
   * [csv](https://salgo60.github.io/SCB-Wikidata/notebook/resultsRiksdagenWD/links_RiksdagenWikidata_v1_2026_02_08.csv)
 


In [1]:
import time

from datetime import datetime

now = datetime.now()
timestamp = now.timestamp()

start_time = time.time()
print("Start:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Start: 2026-02-16 01:31:25


In [2]:
WikidataInclude = False
SCRIPT_NAME = "riksdagen_64_wikidata.ipynb"
SCRIPT_URL = (
    "https://github.com/salgo60/SCB-Wikidata/"
    "blob/master/notebook/riksdagen_64_wikidata.ipynb"
) 


In [3]:
import os

# Get the current working directory
current_directory = os.getcwd()
print("Current Working Directory:", current_directory)



Current Working Directory: /Users/salgo/Documents/GitHub/SCB-Wikidata/notebook


In [4]:
import requests

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "Mozilla/5.0 (compatible; URL-checker/1.0) salgo60@msn.com"
})


In [5]:
def read_domains(file_path):
    print(f"[DEBUG] Reading domains from: {file_path}")
    df = pd.read_csv(file_path, header=0)   # <- skip header row
    domains_list = df.iloc[:, 0].dropna().unique().tolist()
    print(f"[DEBUG] Found {len(domains_list)} domains.")
    return domains_list


In [6]:
import requests

def fetch_sitematrix_df():
    url = "https://meta.wikimedia.org/w/api.php"
    params = {
        "action": "sitematrix",
        "format": "json"
    }
    headers = {
        "User-Agent": "salgo60-language-fetcher/1.0 (salgo60@msn.com)"
    }

    print("[DEBUG] Fetching sitematrix…")
    r = requests.get(url, params=params, headers=headers)
    r.raise_for_status()

    if "application/json" not in r.headers.get("Content-Type", ""):
        raise ValueError("Server returned non-JSON response")

    data = r.json()["sitematrix"]

    rows = []

    # --- language-specific sites ---
    for key, lang_block in data.items():
        if not key.isdigit():
            continue  # skip "count", "specials"

        lang_code = lang_block.get("code")
        lang_name = lang_block.get("name")

        for site in lang_block.get("site", []):
            rows.append({
                "lang_code": lang_code,
                "lang_name": lang_name,
                "project": site.get("project"),
                "url": site.get("url"),
                "dbname": site.get("dbname"),
                "site_name": site.get("sitename"),
                "closed": site.get("closed", False)
            })

    # --- special wikis (Wikidata, Commons, Meta, etc.) ---
    for site in data.get("specials", []):
        rows.append({
            "lang_code": "special",
            "lang_name": "special",
            "project": site.get("project"),
            "url": site.get("url"),
            "dbname": site.get("dbname"),
            "site_name": site.get("sitename"),
            "closed": site.get("closed", False)
        })

    return pd.DataFrame(rows)


In [7]:
import requests
import pandas as pd


HEADERS = {
    "User-Agent": "salgo60-language-fetcher/2.0 (https://github.com/salgo60) salgo60@msn.com"
}


df_lang_fetch = fetch_sitematrix_df()
df_lang_fetch["closed"] = df_lang_fetch["closed"].fillna(False).astype(bool)

df_lang_wikipedia = df_lang_fetch[
    (df_lang_fetch["site_name"] == "Wikipedia") &
    (
        (df_lang_fetch["lang_name"].str.lower() != "special") |
        (df_lang_fetch["dbname"] == "wikidatawiki")
    )
]  

#df_lang_wikipedia.to_csv("test.csv")
df_lang_wikipedia.info()

[DEBUG] Fetching sitematrix…
<class 'pandas.core.frame.DataFrame'>
Index: 186 entries, 0 to 1047
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   lang_code  186 non-null    object
 1   lang_name  185 non-null    object
 2   project    0 non-null      object
 3   url        186 non-null    object
 4   dbname     186 non-null    object
 5   site_name  186 non-null    object
 6   closed     186 non-null    bool  
dtypes: bool(1), object(6)
memory usage: 10.4+ KB


In [8]:
df_lang_wikipedia

,lang_code,lang_name,project,url,dbname,site_name,closed
0,aa,Qafár af,None,https://aa.wikipedia.org,aawiki,Wikipedia,False
5,ace,Acèh,None,https://ace.wikipedia.org,acewiki,Wikipedia,False
7,af,Afrikaans,None,https://af.wikipedia.org,afwiki,Wikipedia,False
11,ak,None,None,https://ak.wikipedia.org,akwiki,Wikipedia,False
18,ami,Pangcah,None,https://ami.wikipedia.org,amiwiki,Wikipedia,False
...,...,...,...,...,...,...,...
924,za,Vahcuengh,None,https://za.wikipedia.org,zawiki,Wikipedia,False
928,zea,Zeêuws,None,https://zea.wikipedia.org,zeawiki,Wikipedia,False
931,zh,中文,None,https://zh.wikipedia.org,zhwiki,Wikipedia,False
939,zu,isiZulu,None,https://zu.wikipedia.org,zuwiki,Wikipedia,False


### Bara köra Wikidata 

In [9]:
df_wd = df_lang_wikipedia[df_lang_wikipedia["lang_code"] == "special"]

In [10]:
df_wd

,lang_code,lang_name,project,url,dbname,site_name,closed
1047,special,special,None,https://www.wikidata.org,wikidatawiki,Wikipedia,False


In [11]:
def read_domains(file_path):
    print(f"[DEBUG] Reading domains from: {file_path}")
    df = pd.read_csv(file_path, header=0)   # <- skip header row
    domains_list = df.iloc[:, 0].dropna().unique().tolist()
    print(f"[DEBUG] Found {len(domains_list)} domains.")
    return domains_list


In [12]:
import os
import time
import random
import requests
import pandas as pd
from urllib.parse import urlparse
from tqdm.notebook import tqdm
file_path_domain = "sources/domains_riksdagen.csv"
domains = read_domains(file_path_domain)
print(domains)


[DEBUG] Reading domains from: sources/domains_riksdagen.csv
[DEBUG] Found 1 domains.
['riksdagen.se']


In [13]:
import requests

def fetch_sitematrix_df():
    url = "https://meta.wikimedia.org/w/api.php"
    params = {
        "action": "sitematrix",
        "format": "json"
    }
    headers = {
        "User-Agent": "salgo60-language-fetcher/1.0 (salgo60@msn.com)"
    }

    print("[DEBUG] Fetching sitematrix…")
    r = requests.get(url, params=params, headers=headers)
    r.raise_for_status()

    if "application/json" not in r.headers.get("Content-Type", ""):
        raise ValueError("Server returned non-JSON response")

    data = r.json()["sitematrix"]

    rows = []


    # --- special wikis (Wikidata, Commons, Meta, etc.) ---
    for site in data.get("specials", []):
        print(site)
        rows.append({
            "lang_code": "special",
            "lang_name": "special",
            "project": site.get("project"),
            "url": site.get("url"),
            "dbname": site.get("dbname"),
            "site_name": site.get("sitename"),
            "closed": site.get("closed", False)
        })

    return pd.DataFrame(rows)


In [14]:

df_wd.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1 entries, 1047 to 1047
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   lang_code  1 non-null      object
 1   lang_name  1 non-null      object
 2   project    0 non-null      object
 3   url        1 non-null      object
 4   dbname     1 non-null      object
 5   site_name  1 non-null      object
 6   closed     1 non-null      bool  
dtypes: bool(1), object(6)
memory usage: 57.0+ bytes


In [15]:
df_wd

,lang_code,lang_name,project,url,dbname,site_name,closed
1047,special,special,None,https://www.wikidata.org,wikidatawiki,Wikipedia,False


In [16]:
def resolve_api_base(lang):
    if lang == "special":
        # Wikidata (only valid "special" case in your pipeline)
        return "https://www.wikidata.org/w/api.php"

    return f"https://{lang}.wikipedia.org/w/api.php"


In [17]:
def request_timeout(lang):
    return 100 if lang == "special" else 10

In [18]:
import os

CHECKPOINT_FILE = "checkpoint_riksdagen_wd_links.parquet"

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        df_cp = pd.read_parquet(CHECKPOINT_FILE)
        print(f"Checkpoint hittad: {len(df_cp)} rader")
        return df_cp
    return pd.DataFrame()


In [19]:
# -----------------------------------------------------------
# Fetch exturlusage entries for one lang/domain
# -----------------------------------------------------------
def fetch_exturlusage(lang, domain):
    #base = f"https://{lang}.wikipedia.org/w/api.php"
    base = resolve_api_base(lang)
    params = {
        "action": "query",
        "format": "json",
        "list": "exturlusage",
        "euquery": domain,
        "eulimit": "max"
    }
    while True:
        #r = session.get(base, params=params, timeout=10)
        r = session.get(base, params=params, timeout=request_timeout(lang))        
        try:
            data = r.json()
        except ValueError:
            print(f"[WARN] {lang}: JSON decode failed")
            break

        for item in data.get("query", {}).get("exturlusage", []):
            yield {
                "lang": lang,
                "page_title": item.get("title"),
                "url": item.get("url"),
                "wiki_link": f"https://{lang}.wikipedia.org/wiki/{item.get('title').replace(' ', '_')}"
            }

        if "continue" not in data:
            break
        params.update(data["continue"])
        time.sleep(0.3)

In [20]:
df_wd

,lang_code,lang_name,project,url,dbname,site_name,closed
1047,special,special,None,https://www.wikidata.org,wikidatawiki,Wikipedia,False


In [21]:
domains

['riksdagen.se']

In [22]:
UseWDCache = True  # True = använd cache, False = hämta från WD
CACHE_FILE = "riksdagen_wd_cache.json"

In [23]:
import json
import os

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        cache = json.load(f)
else:
    cache = {}


In [24]:
def fetch_with_cache(lang, domains):
    global cache

    key = f"{lang}|{'|'.join(domains)}"

    # --- Läs från cache ---
    if UseWDCache:
        if key in cache:
            print("CACHE HIT:", key)
            return cache[key]
        else:
            print("CACHE MISS (UseWDCache=True):", key)
            return []

    # --- Hämta från WD ---
    data = list(fetch_exturlusage(lang, domains))

    cache[key] = data

    # skriv till fil direkt (säkert vid långa körningar)
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(cache, f)

    return data


In [25]:

# -------------------------
# Session & helpers
# -------------------------
session = requests.Session()
session.headers.update({"User-Agent": "SCB-LinkAudit/1.0 salgo60@msn.com"})


print("Antal Språk:",len(df_wd))

results = []



results = []

for _, row in df_wd.iterrows():
    lang = row["lang_code"]
    url  = row["url"]
    lang_name = row["lang_name"]

    before = len(results)

    print(lang, url, lang_name, domains)

    entries = fetch_with_cache(lang, domains)
    results.extend(entries)

    after = len(results)
    print(lang, url, lang_name, "-", after-before)


Antal Språk: 1
special https://www.wikidata.org special ['riksdagen.se']
CACHE HIT: special|riksdagen.se
special https://www.wikidata.org special - 617191


In [26]:
df_riksdagen_wd = pd.DataFrame(results)
df_riksdagen_wd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 617191 entries, 0 to 617190
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   lang        617191 non-null  object
 1   page_title  617191 non-null  object
 2   url         617191 non-null  object
 3   wiki_link   617191 non-null  object
dtypes: object(4)
memory usage: 18.8+ MB


In [27]:
import pandas as pd

# --- Stats ---
total_links = len(df_riksdagen_wd)
total_unique_links = df_riksdagen_wd['url'].nunique()
num_languages = df_riksdagen_wd['lang'].nunique()
langs_sorted = df_riksdagen_wd['lang'].value_counts()

print("Total links:", total_links)
print("Total unique links:", total_unique_links)
print("Wikidata")


Total links: 617191
Total unique links: 614687
Wikidata


### Check url

In [28]:
import requests 
from requests.exceptions import RequestException 
from concurrent.futures import ThreadPoolExecutor, as_completed 
from tqdm import tqdm 
import pandas as pd 
import time  
import threading

In [29]:
MAX_WORKERS = 64              # För IO-bound scraping kan du ofta köra 32–128 workers
MAX_WORKERS = min(128, os.cpu_count()*10)
REQUEST_TIMEOUT = 15

RATE_LIMIT_PER_SEC = 2       # max requests/sek totalt
BACKOFF_FACTOR = 2
MAX_RETRIES = 3

CHECKPOINT_EVERY = 500
CHUNK_SIZE = 5000

CHECKPOINT_FILE = "checkpoint_riksdagen_wikidata.parquet"


In [30]:
import threading
import time

rate_lock = threading.Lock()
last_request_time = 0

def wait_for_rate_limit():
    global last_request_time
    
    with rate_lock:
        now = time.time()
        wait = 1 / RATE_LIMIT_PER_SEC - (now - last_request_time)
        if wait > 0:
            time.sleep(wait)
        last_request_time = time.time()


In [31]:
def request_with_retry(method, url, session, **kwargs):
    delay = 1
    
    for attempt in range(MAX_RETRIES):
        try:
            wait_for_rate_limit()
            return session.request(method, url, **kwargs)
        
        except RequestException:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(delay)
            delay *= BACKOFF_FACTOR


In [32]:
def check_url(url: str) -> dict:
    session = get_session()

    try:
        r = request_with_retry(
            "HEAD",
            url,
            session,
            allow_redirects=False,
            timeout=REQUEST_TIMEOUT,
        )
    except RequestException as e:
        return {"url": url, "status": "error", "reason": str(e)}

    final_url = r.url

    if r.status_code >= 400:
        return {"url": url, "status": "dead", "reason": f"HTTP {r.status_code}"}

    if norm(final_url) == norm(ROOT_CANONICAL) and norm(url) != norm(final_url):
        return {"url": url, "status": "dead", "reason": "redirect_to_root"}

    try:
        r = request_with_retry(
            "GET",
            final_url,
            session,
            allow_redirects=True,
            timeout=REQUEST_TIMEOUT,
        )
    except RequestException as e:
        return {"url": url, "status": "error", "reason": str(e)}

    if looks_like_soft_404(r):
        return {"url": url, "status": "dead", "reason": "soft_404"}

    return {"url": url, "status": "ok"}
    

In [33]:
import os

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        return pd.read_parquet(CHECKPOINT_FILE)
    return pd.DataFrame()


In [34]:
from itertools import islice

def chunks(iterable, size):
    it = iter(iterable)
    while chunk := list(islice(it, size)):
        yield chunk


In [35]:
# ========================== # Internet Archive # ========================== 
def check_internet_archive(url: str) -> str | None: 
    session = get_session() 
    api = "https://archive.org/wayback/available" 
    
    try: 
        r = session.get( api, 
                        params={"url": url}, 
                        timeout=10, ) 
        data = r.json() 
    except Exception: 
        return None 
    snap = data.get("archived_snapshots", {}).get("closest") 
    if snap and snap.get("available"): 
        return snap.get("url") 
    return None

In [36]:


def is_worth_checking(url: str) -> bool: 
    if not url.startswith("http"):
        return False 
    if "riksdagen.se" not in url: 
        return False 
    return True

In [37]:
# ========================== # Worker # ========================== 
def process_url(url: str) -> dict: 
    result = check_url(url) 
    if result["status"] == "dead": 
        ia_url = check_internet_archive(url) 
        result["ia_url"] = ia_url 
        result["ia_status"] = "available" if ia_url else "missing" 
    else: 
        result["ia_url"] = None 
        result["ia_status"] = "skipped" 
    return result

In [38]:
def run(df):

    # -------------------------
    # 1. Bygg unik URL-lista
    # -------------------------
    urls_all = (
        df["url"]
            .dropna()
            .astype(str)
            .unique()
    )

    urls_all = [u for u in urls_all if is_worth_checking(u)]

    # -------------------------
    # 2. Ladda och städa checkpoint
    # -------------------------
    cp = load_checkpoint()

    if not cp.empty:
        cp = (
            cp
                .sort_index()
                .drop_duplicates(subset="url", keep="last")
                .reset_index(drop=True)
        )

    done = set(cp["url"]) if not cp.empty else set()

    urls_remaining = [u for u in urls_all if u not in done]

    print("Totalt:", len(urls_all))
    print("Redan klara:", len(done))
    print("Kvar:", len(urls_remaining))

    if not urls_remaining:
        print("Inget att göra — allt redan processat.")
        return cp

    # -------------------------
    # 3. Processa i chunkar
    # -------------------------
    for chunk_idx, chunk in enumerate(chunks(urls_remaining, CHUNK_SIZE), 1):

        print(f"\nStartar chunk {chunk_idx} ({len(chunk)} URL:er)")

        results_chunk = []

        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = {ex.submit(process_url, u): u for u in chunk}

            for f in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"Chunk {chunk_idx}",
            ):
                res = f.result()
                url = res["url"]

                # SKYDD 1: dubbelkörningsskydd
                if url in done:
                    continue

                results_chunk.append(res)
                done.add(url)

        # -------------------------
        # 4. Spara chunk direkt
        # -------------------------
        if results_chunk:

            df_chunk = pd.DataFrame(results_chunk)

            if os.path.exists(CHECKPOINT_FILE):
                old = pd.read_parquet(CHECKPOINT_FILE)
                df_new = (
                    pd.concat([old, df_chunk])
                        .drop_duplicates(subset="url", keep="last")
                )
            else:
                df_new = df_chunk

            tmp = CHECKPOINT_FILE + ".tmp"
            df_new.to_parquet(tmp)
            os.replace(tmp, CHECKPOINT_FILE)

            print(f"Checkpoint uppdaterad ({len(df_new)} unika)")

    # -------------------------
    # 5. Slutresultat
    # -------------------------
    final = pd.read_parquet(CHECKPOINT_FILE)
    final = final.drop_duplicates(subset="url", keep="last")

    print("\nKLAR.")
    print("Totalt unika:", len(final))

    return final

In [39]:

import requests
import threading

USER_AGENT = "LinkChecker/1.0 salgo60@msn.com"

thread_local = threading.local()

def get_session():
    if not hasattr(thread_local, "session"):
        s = requests.Session()
        s.headers.update({"User-Agent": USER_AGENT})
        thread_local.session = s
    return thread_local.session


def norm(u: str) -> str:
    return u.rstrip("/").lower()


def is_worth_checking(url: str) -> bool:
    return url.startswith("http") and "riksdagen.se" in url



In [40]:
def looks_like_soft_404(response) -> bool:
    text = (response.text or "").lower()

    for phrase in SOFT_404_PHRASES:
        if phrase in text:
            return True

    if "<title>" in text:
        title = text.split("<title>", 1)[1].split("</title>", 1)[0]
        if "404" in title.lower():
            return True
        if "sidan finns inte" in title.lower():
            return True

    return False

SOFT_404_PHRASES = [
    "sidan kan inte hittas",
    "sidan tagits bort",
    "felaktig adress",
    "kontakta registrator",
    "sidan finns inte",
    "den här sidan kan inte visas",
    "Hoppsan! Vi kunde tyvärr inte hitta sidan"
]

USER_AGENT = "LinkChecker/1.0 (research; salgo60@msn.com)"


In [41]:
ROOT_CANONICAL = "https://www.riksdagen.se"

In [42]:
df_result = run(df_riksdagen_wd)

from datetime import date
import os

# Sätt datum
today = date.today().strftime("%Y_%m_%d")

# Se till att katalogen finns
os.makedirs("resultsRiksdagenWD", exist_ok=True)

# Bygg filnamn
outfile = f"resultsRiksdagenWD/links_Riksdagen_v1_Wikidata_{today}.csv"
    
# Exportera
df_riksdagen_wd.to_csv(outfile, index=False, encoding="utf-8")

print(f"[OK] Exported {len(df_riksdagen_wd)} rows to {outfile}")


Totalt: 614687
Redan klara: 317500
Kvar: 297187
MAX_WORKERS: 80

Startar chunk 1 (5000 URL:er)

Startar chunk 1 (5000 URL:er)


Chunk 1:  10%|██▉                          | 499/5000 [23:07<3:34:35,  2.86s/it]

318000 totalt klara...


Chunk 1:  10%|██▉                          | 500/5000 [23:15<5:33:00,  4.44s/it]

Checkpoint sparad (318000)


...

661000 totalt klara...


Chunk 10:  26%|██████▌                  | 1313/5000 [2:18:55<1:27:17,  1.42s/it]

Checkpoint sparad (661000)


Chunk 10:  30%|███████▍                 | 1489/5000 [2:22:05<5:35:02,  5.73s/it]


KeyboardInterrupt: 

In [43]:
cp = load_checkpoint()

print("Checkpoint rows:", len(cp))
print("Unika URL i checkpoint:", cp["url"].nunique())
print("Dubletter i checkpoint:", len(cp) - cp["url"].nunique())

Checkpoint rows: 661000
Unika URL i checkpoint: 614687
Dubletter i checkpoint: 46313


In [44]:
cp = load_checkpoint()

cp_clean = (
    cp.sort_index()
      .drop_duplicates(subset="url", keep="last")
)

print("Efter dedupe:", len(cp_clean))
print("Unika:", cp_clean["url"].nunique())

cp_clean.to_parquet("final.parquet")

Efter dedupe: 614687
Unika: 614687


In [49]:
cp = load_checkpoint()

print("Rader:", len(cp))
print("Unika URL:", cp["url"].nunique())

cp_clean = (
    cp
        .sort_index()
        .drop_duplicates(subset="url", keep="last")
        .reset_index(drop=True)
)

print("Efter dedupe:", len(cp_clean))

Rader: 661000
Unika URL: 614687
Efter dedupe: 614687


In [50]:
urls_wd = (
    df_riksdagen_wd["url"]
        .dropna()
        .astype(str)
        .unique()
)

print("WD unika:", len(urls_wd))
print("Checkpoint unika:", cp_clean["url"].nunique())

missing = set(urls_wd) - set(cp_clean["url"])
print("Saknade:", len(missing))

WD unika: 614687
Checkpoint unika: 614687
Saknade: 0


In [51]:
df_final = (
    df_riksdagen_wd
        .merge(
            cp_clean[["url", "status", "reason", "ia_url", "ia_status"]],
            on="url",
            how="left"
        )
)

In [52]:
print("WD rows:", len(df_riksdagen_wd))
print("Final rows:", len(df_final))

print("Status saknas:", df_final["status"].isna().sum())

WD rows: 617191
Final rows: 617191
Status saknas: 0


In [54]:
from datetime import date
import os

# Sätt datum
today = date.today().strftime("%Y_%m_%d")

# Se till att katalogen finns
os.makedirs("resultsRiksdagenWD", exist_ok=True)

# Bygg filnamn
outfile = f"resultsRiksdagenWD/links_Riksdagen_v1_Wikidata_{today}.csv"
    

In [55]:
df_final.to_csv(outfile, index=False, encoding="utf-8") 
print(f"[OK] Exported {len(df_riksdagen_wd)} rows to {outfile}")


[OK] Exported 617191 rows to resultsRiksdagenWD/links_Riksdagen_v1_Wikidata_2026_02_21.csv


In [46]:
cp_clean["url"].isna().sum()

0

In [56]:
def classify_link(row):
    if row["status"] == "ok":
        return "ok"
    if row["status"] == "dead":
        if row.get("ia_status") == "available":
            return "broken_archived"
        else:
            return "broken_lost"
    if row["status"] == "error":
        return "error"
    return "unknown"

df_final["link_status"] = df_final.apply(classify_link, axis=1)

In [57]:
df_final["link_status"].value_counts()

link_status
ok                 597816
broken_archived      9143
broken_lost          7159
error                3073
Name: count, dtype: int64

In [58]:
lang_stats = (
    df_final
        .groupby("lang")
        .agg(
            total_links=("url", "count"),
            broken_total=("link_status", lambda s: s.str.startswith("broken").sum()),
            broken_lost=("link_status", lambda s: (s == "broken_lost").sum()),
            archived=("link_status", lambda s: (s == "broken_archived").sum()),
            errors=("link_status", lambda s: (s == "error").sum()),
        )
        .reset_index()
)

lang_stats["broken_pct"] = (
    100 * lang_stats["broken_total"] / lang_stats["total_links"]
).round(1)

In [59]:
lang_stats

,lang,total_links,broken_total,broken_lost,archived,errors,broken_pct
0,special,617191,16302,7159,9143,3073,2.6


In [47]:
lang_stats = (
    df_riksdagen_wd
    .groupby("lang")
    .agg(
        total_links=("url", "count"),
        broken_links=("status", lambda s: (s == "dead").sum()),
        archived_links=("ia_status", lambda s: (s == "available").sum()),
    )
    .reset_index()
)

lang_stats["broken_pct"] = (
    100 * lang_stats["broken_links"] / lang_stats["total_links"]
).round(1)

lang_stats["broken_lost"] = (
    lang_stats["broken_links"] - lang_stats["archived_links"])

top10_langs = (
    lang_stats
    .sort_values("total_links", ascending=False)
    .head(10)
)
top10_langs[
    [
        "lang",
        "total_links",
        "broken_links",
        "broken_pct",
        "archived_links",
        "broken_lost",
    ]
]

KeyError: "Column(s) ['ia_status', 'status'] do not exist"

In [ ]:
from urllib.parse import urlparse

df = df_riksdagen_wd.copy()

df["domain"] = df["url"].apply(
    lambda u: urlparse(u).netloc.lower() if pd.notna(u) else None
)
domain_stats = (
    df
    .groupby("domain")
    .agg(
        total_links=("url", "count"),
        broken_links=("status", lambda s: (s == "dead").sum()),
        error_links=("status", lambda s: (s == "error").sum()),
    )
    .reset_index()
)
domain_stats["broken_pct"] = (
    100 * domain_stats["broken_links"] / domain_stats["total_links"]
).round(1)

domain_stats["error_pct"] = (
    100 * domain_stats["error_links"] / domain_stats["total_links"]
).round(1)


In [ ]:
status_counts = df["status"].value_counts()

num_ok = int(status_counts.get("ok", 0))
num_dead = int(status_counts.get("dead", 0))
num_error = int(status_counts.get("error", 0))
num_total = len(df)

pct_ok = round(100 * num_ok / num_total, 1)
pct_dead = round(100 * num_dead / num_total, 1)
pct_error = round(100 * num_error / num_total, 1)

# Broken links: archived vs lost
num_dead_archived = df[
    (df["status"] == "dead") & (df["ia_status"] == "available")
].shape[0]

num_dead_lost = num_dead - num_dead_archived


In [ ]:
top_domains = (
    domain_stats[domain_stats["total_links"] >= 5]
    .sort_values("total_links", ascending=False)
    .head(10)
)
domain_stats_html = "<ul>"
for _, r in top_domains.iterrows():
    domain_stats_html += (
        f"<li><strong>{r['domain']}</strong>: "
        f"{r['broken_links']} / {r['total_links']} broken "
        f"({r['broken_pct']}%)</li>"
    )
domain_stats_html += "</ul>"


In [62]:
def save_problem_report_wikidata_riksdagen(
    df,
    out_dir="resultsRiksdagenWD",
    issue_url="https://github.com/salgo60/SCB-Wikidata/issues/64",
):

    from pathlib import Path
    from datetime import date, datetime
    import pandas as pd
    from urllib.parse import urlparse, quote

    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)

    today = date.today().strftime("%Y_%m_%d")
    out_path = out_dir / f"wikidata_riksdagen_problem_report_{today}.html"
    rerun_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    df = df.copy()

    # ------------------------------------------------
    # 1. Klassificera problem
    # ------------------------------------------------
    def classify(row):
        if row["status"] == "ok":
            return "ok"
        if row["status"] == "dead" and row.get("ia_status") == "available":
            return "broken_archived"
        if row["status"] == "dead":
            return "broken_lost"
        if row["status"] == "error":
            return "error"
        return "unknown"

    df["link_status"] = df.apply(classify, axis=1)

    # Endast problem
    df_problems = df[df["link_status"] != "ok"].copy()

    num_total = len(df)
    num_problem = len(df_problems)

    counts = df["link_status"].value_counts()

    num_archived = counts.get("broken_archived", 0)
    num_lost = counts.get("broken_lost", 0)
    num_error = counts.get("error", 0)

    pct_problem = round(100 * num_problem / num_total, 2)

    # ------------------------------------------------
    # 2. Severity
    # ------------------------------------------------
    severity_map = {
        "broken_lost": "critical",
        "broken_archived": "warning",
        "error": "technical"
    }

    df_problems["severity"] = df_problems["link_status"].map(severity_map)

    df_problems = df_problems.sort_values(
        ["severity"],
        ascending=[True]
    )

    # ------------------------------------------------
    # 3. Orsaksanalys
    # ------------------------------------------------
    reason_stats = (
        df_problems
        .groupby("reason")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    reason_table_html = reason_stats.to_html(
        classes="pivot",
        index=False,
        border=0
    )

    # ------------------------------------------------
    # 4. Domänanalys
    # ------------------------------------------------
    df_problems["domain"] = df_problems["url"].apply(
        lambda u: urlparse(u).netloc.lower() if pd.notna(u) else None
    )

    domain_stats = (
        df_problems
        .groupby("domain")
        .size()
        .reset_index(name="problem_count")
        .sort_values("problem_count", ascending=False)
    )

    domain_table_html = domain_stats.head(20).to_html(
        classes="pivot",
        index=False,
        border=0
    )

    # ------------------------------------------------
    # 5. HTML-tabell (endast problem)
    # ------------------------------------------------
    html_table = df_problems.to_html(
        classes="pivot",
        border=0,
        escape=False,
        index=False,
    )

    # ------------------------------------------------
    # 6. Executive summary
    # ------------------------------------------------
    meta_html = f"""
    <div class="meta">
      <h2>Problem Summary</h2>

      <p><strong>Run:</strong> {rerun_ts}</p>

      <p>
        <strong>Total links analyzed:</strong> {num_total}<br>
        <strong>Total problem links:</strong> {num_problem} ({pct_problem}%)<br><br>

        <strong style="color:#b71c1c;">Broken – lost:</strong> {num_lost}<br>
        <strong style="color:#c62828;">Broken – archived:</strong> {num_archived}<br>
        <strong style="color:#ef6c00;">Technical errors:</strong> {num_error}
      </p>

      <p>
        This report focuses on links stored in Wikidata that do not
        currently resolve to accessible content.
      </p>

      <p>
        The presence of unresolved links may indicate:
      </p>

      <ul>
        <li>Identifier instability</li>
        <li>Missing redirect policies</li>
        <li>Absence of tombstone pages</li>
        <li>Structural URL changes</li>
      </ul>

      <p><strong>Reference issue:</strong>
         <a href="{issue_url}" target="_blank">
         {issue_url.split("/")[-1]}</a>
      </p>
    </div>
    """

    # ------------------------------------------------
    # 7. Slutlig HTML
    # ------------------------------------------------
    html = f"""
    <html>
    <head>
      <meta charset="utf-8">
      <title>Wikidata – Riksdagen link integrity report</title>
      <style>
        body {{ font-family: Arial; margin: 20px; }}
        table.pivot {{ border-collapse: collapse; width: 100%; font-size: 12px; }}
        table.pivot th, table.pivot td {{
            border: 1px solid #999;
            padding: 6px;
        }}
        table.pivot th {{
            background: #f2f2f2;
        }}
        .meta {{
            background: #f8f8f8;
            border: 1px solid #ccc;
            padding: 12px;
            margin-bottom: 20px;
        }}
      </style>
    </head>
    <body>

      <h1>Wikidata → Riksdagen link integrity assessment</h1>

      {meta_html}

      <h2>Problem breakdown by cause</h2>
      {reason_table_html}

      <h2>Domains most affected</h2>
      {domain_table_html}

      <h2>Detailed problem list</h2>
      {html_table}

    </body>
    </html>
    """

    out_path.write_text(html, encoding="utf-8")
    print(f"✅ Problem report created: {out_path}")

In [63]:
save_problem_report_wikidata_riksdagen(df_final)

✅ Problem report created: resultsRiksdagenWD/wikidata_riksdagen_problem_report_2026_02_21.html


### Testa om errors som bl.a har timeout 

In [64]:
df_errors = df_final[df_final["status"] == "error"].copy()

print("Antal error:", len(df_errors))

Antal error: 3073


In [65]:
urls_retry = df_errors["url"].unique().tolist()

In [66]:
# Sänk workers
MAX_WORKERS = 3
REQUEST_TIMEOUT = 30
RATE_LIMIT_PER_SEC = 1

In [67]:
def retry_errors(urls):

    results = []

    with ThreadPoolExecutor(max_workers=12) as ex:
        futures = {ex.submit(process_url, u): u for u in urls}

        for f in tqdm(as_completed(futures), total=len(futures)):
            res = f.result()
            results.append(res)

    return pd.DataFrame(results)

In [68]:
df_retry = retry_errors(urls_retry)

100%|█████████████████████████████████████| 3066/3066 [2:25:11<00:00,  2.84s/it]


In [69]:
df_retry = df_retry.drop_duplicates(subset="url", keep="last")

df_final_updated = df_final.merge(
    df_retry[["url", "status", "reason", "ia_url", "ia_status"]],
    on="url",
    how="left",
    suffixes=("", "_retry")
)

# Om retry gav bättre status → ersätt
mask = df_final_updated["status"] == "error"

for col in ["status", "reason", "ia_url", "ia_status"]:
    df_final_updated.loc[mask, col] = df_final_updated.loc[mask, f"{col}_retry"]

In [77]:
def save_problem_report_wikidata_riksdagen(
    df,
    out_dir="resultsRiksdagenWD",
    issue_url="https://github.com/salgo60/SCB-Wikidata/issues/64",
):

    from pathlib import Path
    from datetime import date, datetime
    import pandas as pd
    from urllib.parse import urlparse

    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)

    today = date.today().strftime("%Y_%m_%d")
    out_path = out_dir / f"wikidata_riksdagen_problem_report_{today}.html"
    rerun_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    df = df.copy()

    # ------------------------------------------------
    # Klassificering
    # ------------------------------------------------
    def classify(row):
        if row["status"] == "ok":
            return "ok"
        if row["status"] == "dead" and row.get("ia_status") == "available":
            return "broken_archived"
        if row["status"] == "dead":
            return "broken_lost"
        if row["status"] == "error":
            return "error"
        return "unknown"

    df["link_status"] = df.apply(classify, axis=1)

    df_problems = df[df["link_status"] != "ok"].copy()

    num_total = len(df)
    num_problem = len(df_problems)

    counts = df["link_status"].value_counts()
    num_archived = counts.get("broken_archived", 0)
    num_lost = counts.get("broken_lost", 0)
    num_error = counts.get("error", 0)

    pct_problem = round(100 * num_problem / num_total, 2)

    # ------------------------------------------------
    # Orsaksanalys
    # ------------------------------------------------
    reason_stats = (
        df_problems
        .groupby("reason")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    reason_table_html = reason_stats.to_html(
        classes="pivot",
        index=False,
        border=0
    )

    # ------------------------------------------------
    # Render Wikidata Q-id som länk
    # ------------------------------------------------
    if "page_title" in df_problems.columns:
        df_problems.rename(columns={"page_title": "Wikidata"}, inplace=True)

        df_problems["Wikidata"] = df_problems["Wikidata"].apply(
            lambda q: f'<a href="https://www.wikidata.org/wiki/{q}" target="_blank">{q}</a>'
            if pd.notna(q) and str(q).startswith("Q")
            else q
        )

    # ------------------------------------------------
    # Gör url klickbar
    # ------------------------------------------------
    df_problems["url"] = df_problems["url"].apply(
        lambda u: f'<a href="{u}" target="_blank">{u}</a>'
        if pd.notna(u)
        else ""
    )

    # ------------------------------------------------
    # IA ikon
    # ------------------------------------------------
    def render_ia_icon(row):
        if row.get("ia_status") == "available" and row.get("ia_url"):
            return (
                f'<a href="{row["ia_url"]}" target="_blank" '
                f'title="Archived copy">'
                f'📦</a>'
            )
        return ""

    df_problems["Archive"] = df_problems.apply(render_ia_icon, axis=1)

    # Ta bort kolumner som inte behövs
    drop_cols = ["lang", "wiki_link"]
    for col in drop_cols:
        if col in df_problems.columns:
            df_problems.drop(columns=[col], inplace=True)

    # ------------------------------------------------
    # HTML-tabell
    # ------------------------------------------------
    html_table = df_problems.to_html(
        classes="pivot",
        border=0,
        escape=False,
        index=False,
    )

    # ------------------------------------------------
    # Executive summary
    # ------------------------------------------------
    meta_html = f"""
    <div class="meta">
      <h2>Executive Summary</h2>

      <p><strong>Run:</strong> {rerun_ts}</p>

      <p>
        <strong>Total links analyzed:</strong> {num_total}<br>
        <strong>Links with issues:</strong> {num_problem} ({pct_problem}%)<br><br>

        <strong style="color:#b71c1c;">Broken – lost:</strong> {num_lost}<br>
        <strong style="color:#c62828;">Broken – archived:</strong> {num_archived}<br>
        <strong style="color:#ef6c00;">Technical errors:</strong> {num_error}
      </p>

      <p>
        This report lists Wikidata items whose stored riksdagen.se links
        do not currently resolve to accessible content.
      </p>

      <p><strong>Reference issue:</strong>
         <a href="{issue_url}" target="_blank">
         {issue_url.split("/")[-1]}</a>
      </p>
    </div>
    """

    # ------------------------------------------------
    # HTML-dokument
    # ------------------------------------------------
    html = f"""
    <html>
    <head>
      <meta charset="utf-8">
      <title>Wikidata – Riksdagen link integrity</title>
      <style>
        body {{ font-family: Arial; margin: 20px; }}
        table.pivot {{ border-collapse: collapse; width: 100%; font-size: 12px; }}
        table.pivot th, table.pivot td {{
            border: 1px solid #999;
            padding: 6px;
        }}
        table.pivot th {{ background: #f2f2f2; }}
        .meta {{
            background: #f8f8f8;
            border: 1px solid #ccc;
            padding: 12px;
            margin-bottom: 20px;
        }}
      </style>
    </head>
    <body>

      <h1>Wikidata → Riksdagen non-resolving links</h1>

      {meta_html}

      <h2>Problem breakdown by cause</h2>
      {reason_table_html}

      <h2>Detailed list (non-resolving links only)</h2>
      {html_table}

    </body>
    </html>
    """

    out_path.write_text(html, encoding="utf-8")
    print(f"✅ Problem report created: {out_path}")

In [86]:
def save_problem_report_wikidata_riksdagen(
    df,
    out_dir="resultsRiksdagenWD",
    issue_url="https://github.com/salgo60/SCB-Wikidata/issues/64",
):

    from pathlib import Path
    from datetime import date, datetime
    import pandas as pd
    from urllib.parse import urlparse

    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)

    today = date.today().strftime("%Y_%m_%d")
    out_path = out_dir / f"wikidata_riksdagen_problem_report_{today}.html"
    rerun_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    df = df.copy()

    # ------------------------------------------------
    # Klassificering
    # ------------------------------------------------
    def classify(row):
        if row["status"] == "ok":
            return "ok"
        if row["status"] == "dead" and row.get("ia_status") == "available":
            return "broken_archived"
        if row["status"] == "dead":
            return "broken_lost"
        if row["status"] == "error":
            return "error"
        return "unknown"

    df["link_status"] = df.apply(classify, axis=1)

    df_problems = df[df["link_status"] != "ok"].copy()

    num_total = len(df)
    num_problem = len(df_problems)

    counts = df["link_status"].value_counts()
    num_archived = counts.get("broken_archived", 0)
    num_lost = counts.get("broken_lost", 0)
    num_error = counts.get("error", 0)

    pct_problem = round(100 * num_problem / num_total, 2)

    # ------------------------------------------------
    # Orsaksanalys
    # ------------------------------------------------
    reason_stats = (
        df_problems
        .groupby("reason")
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    reason_table_html = reason_stats.to_html(
        classes="pivot",
        index=False,
        border=0
    )

    from urllib.parse import quote
    # ------------------------------------------------
    # Gör Wikidata-kolumn klickbar (alla typer)
    # ------------------------------------------------
    if "page_title" in df_problems.columns:
        df_problems.rename(columns={"page_title": "Wikidata"}, inplace=True)
    
    if "Wikidata" in df_problems.columns:
        df_problems["Wikidata"] = df_problems["Wikidata"].apply(
            lambda v: (
                f'<a href="https://www.wikidata.org/wiki/{quote(str(v))}" '
                f'target="_blank">{v}</a>'
            )
            if pd.notna(v)
            else ""
        )
    # ------------------------------------------------
    # Gör url klickbar
    # ------------------------------------------------
    df_problems["url"] = df_problems["url"].apply(
        lambda u: f'<a href="{u}" target="_blank">{u}</a>'
        if pd.notna(u)
        else ""
    )

    # ------------------------------------------------
    # IA ikon
    # ------------------------------------------------
    def render_ia_icon(row):
        if row.get("ia_status") == "available" and row.get("ia_url"):
            return (
                f'<a href="{row["ia_url"]}" target="_blank" '
                f'title="Archived copy">'
                f'📦</a>'
            )
        return ""

    df_problems["Archive"] = df_problems.apply(render_ia_icon, axis=1)

    # Ta bort kolumner som inte behövs
    drop_cols = ["lang", "wiki_link"]
    for col in drop_cols:
        if col in df_problems.columns:
            df_problems.drop(columns=[col], inplace=True)
    # ------------------------------------------------
    # IA ikon (ersätter ia_url)
    # ------------------------------------------------
    def render_ia_icon(row):
        if row.get("ia_status") == "available" and row.get("ia_url"):
            return (
                f'<a href="{row["ia_url"]}" target="_blank" '
                f'title="Archived copy (Internet Archive)">'
                f'📦</a>'
            )
        return ""
    
    df_problems["Archive"] = df_problems.apply(render_ia_icon, axis=1)

    # ------------------------------------------------
    # Ta bort råkolumner som inte ska visas
    # ------------------------------------------------
    cols_to_remove = ["ia_url", "ia_status", "severity_rank"]
    
    for col in cols_to_remove:
        if col in df_problems.columns:
            df_problems.drop(columns=[col], inplace=True)
    
    # ------------------------------------------------
    # Välj exakt kolumnordning
    # ------------------------------------------------
    wanted_columns = [
        "Wikidata",
        "url",
        "status",
        "reason",
        "Archive",
        "link_status"
    ]
    
    existing = [c for c in wanted_columns if c in df_problems.columns]
    
    df_problems = df_problems[existing]
    # ------------------------------------------------
    # Ta bort råkolumner som inte ska visas
    # ------------------------------------------------
    cols_to_remove = ["ia_url", "ia_status", "severity_rank"]
    
    for col in cols_to_remove:
        if col in df_problems.columns:
            df_problems.drop(columns=[col], inplace=True)
    
    # ------------------------------------------------
    # Välj exakt kolumnordning
    # ------------------------------------------------
    wanted_columns = [
        "Wikidata",
        "url",
        "status",
        "reason",
        "Archive",
        "link_status"
    ]
    
    existing = [c for c in wanted_columns if c in df_problems.columns]
    
    df_problems = df_problems[existing]
    # ------------------------------------------------
    # HTML-tabell
    # ------------------------------------------------
    html_table = df_problems.to_html(
        classes="pivot",
        border=0,
        escape=False,
        index=False,
    )

    # ------------------------------------------------
    # Executive summary
    # ------------------------------------------------
    meta_html = f"""
    <div class="meta">
      <h2>Executive Summary</h2>

      <p><strong>Run:</strong> {rerun_ts}</p>

      <p>
        <strong>Total links analyzed:</strong> {num_total}<br>
        <strong>Links with issues:</strong> {num_problem} ({pct_problem}%)<br><br>

        <strong style="color:#b71c1c;">Broken – lost:</strong> {num_lost}<br>
        <strong style="color:#c62828;">Broken – archived:</strong> {num_archived}<br>
        <strong style="color:#ef6c00;">Technical errors:</strong> {num_error}
      </p>

      <p>
        This report lists Wikidata items whose stored riksdagen.se links
        do not currently resolve to accessible content.
      </p>

      <p><strong>Reference issue:</strong>
         <a href="{issue_url}" target="_blank">
         {issue_url.split("/")[-1]}</a>
      </p>
    </div>
    """

    # ------------------------------------------------
    # HTML-dokument
    # ------------------------------------------------
    html = f"""
    <html>
    <head>
      <meta charset="utf-8">
      <title>Wikidata – Riksdagen link integrity</title>
      <style>
        body {{ font-family: Arial; margin: 20px; }}
        table.pivot {{ border-collapse: collapse; width: 100%; font-size: 12px; }}
        table.pivot th, table.pivot td {{
            border: 1px solid #999;
            padding: 6px;
        }}
        table.pivot th {{ background: #f2f2f2; }}
        .meta {{
            background: #f8f8f8;
            border: 1px solid #ccc;
            padding: 12px;
            margin-bottom: 20px;
        }}
      </style>
    </head>
    <body>

      <h1>Wikidata → Riksdagen non-resolving links</h1>

      {meta_html}

      <h2>Problem breakdown by cause</h2>
      {reason_table_html}

      <h2>Detailed list (non-resolving links only)</h2>
      {html_table}

    </body>
    </html>
    """

    out_path.write_text(html, encoding="utf-8")
    print(f"✅ Problem report created: {out_path}")

In [94]:
df_problems = df_final_updated[df_final_updated["link_status"] != "ok"] 
df_problems.info() 
df_problems.head()

<class 'pandas.core.frame.DataFrame'>
Index: 19375 entries, 5 to 617185
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   lang             19375 non-null  object
 1   page_title       19375 non-null  object
 2   url              19375 non-null  object
 3   wiki_link        19375 non-null  object
 4   status           19375 non-null  object
 5   reason           16555 non-null  object
 6   ia_url           9306 non-null   object
 7   ia_status        19375 non-null  object
 8   link_status      19375 non-null  object
 9   status_retry     3073 non-null   object
 10  reason_retry     253 non-null    object
 11  ia_url_retry     163 non-null    object
 12  ia_status_retry  3073 non-null   object
dtypes: object(13)
memory usage: 2.1+ MB


,lang,page_title,url,wiki_link,status,reason,ia_url,ia_status,link_status,status_retry,reason_retry,ia_url_retry,ia_status_retry
5,special,Wikidata:Property proposal/Archive/20,http://www.riksdagen.se/sv/ledamoter-partier/H...,https://special.wikipedia.org/wiki/Wikidata:Pr...,dead,HTTP 404,http://web.archive.org/web/20160303234228/http...,available,broken_archived,NaN,NaN,NaN,NaN
6,special,Q2740012,http://www.riksdagen.se/en/Members-and-parties...,https://special.wikipedia.org/wiki/Q2740012,dead,HTTP 404,http://web.archive.org/web/20160303235945/http...,available,broken_archived,NaN,NaN,NaN,NaN
7,special,Q18202407,http://www.riksdagen.se/sv/ledamoter-partier/I...,https://special.wikipedia.org/wiki/Q18202407,dead,HTTP 404,http://web.archive.org/web/20180902025345/http...,available,broken_archived,NaN,NaN,NaN,NaN
8,special,Q2740012,https://www.riksdagen.se/sv/ledamoter-partier/...,https://special.wikipedia.org/wiki/Q2740012,dead,HTTP 404,http://web.archive.org/web/20160407193603/http...,available,broken_archived,NaN,NaN,NaN,NaN
64,special,Property:P1214,http://data.riksdagen.se/Data/Ledamoter,https://special.wikipedia.org/wiki/Property:P1214,dead,HTTP 404,http://web.archive.org/web/20230604005232/http...,available,broken_archived,NaN,NaN,NaN,NaN


In [96]:
import requests
import time

WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"

def fetch_sv_labels_sparql(qids, batch_size=200, sleep=0.2):
    label_map = {}

    headers = {
        "User-Agent": "LinkAudit/1.0 (research; salgo60@msn.com)"
    }

    for i in range(0, len(qids), batch_size):
        batch = qids[i:i+batch_size]

        values = " ".join(f"wd:{qid}" for qid in batch)

        query = f"""
        SELECT ?item ?itemLabel WHERE {{
          VALUES ?item {{ {values} }}
          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "sv,en". }}
        }}
        """

        r = requests.get(
            WIKIDATA_SPARQL,
            params={"query": query, "format": "json"},
            headers=headers,
            timeout=30,
        )

        data = r.json()

        for row in data["results"]["bindings"]:
            qid = row["item"]["value"].split("/")[-1]
            label = row["itemLabel"]["value"]
            label_map[qid] = label

        time.sleep(sleep)

    return label_map

In [98]:
# Extrahera råa Q-id (utan HTML)
df_problems["qid_raw"] = df_problems["page_title"].str.replace(
    r'<.*?>', '', regex=True
)

qids = [
    q for q in df_problems["qid_raw"].dropna().unique()
    if str(q).startswith("Q")
]

print("Unika Q-id:", len(qids))

label_map = fetch_sv_labels_sparql(qids)

df_problems["Titelsv"] = df_problems["qid_raw"].map(label_map)

/var/folders/fd/md6r13sj0wsbg_6_xl160d300000gn/T/ipykernel_5107/313595599.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_problems["qid_raw"] = df_problems["page_title"].str.replace(


Unika Q-id: 17203


/var/folders/fd/md6r13sj0wsbg_6_xl160d300000gn/T/ipykernel_5107/313595599.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_problems["Titelsv"] = df_problems["qid_raw"].map(label_map)


In [99]:
save_problem_report_wikidata_riksdagen(df_final_updated)

✅ Problem report created: resultsRiksdagenWD/wikidata_riksdagen_problem_report_2026_02_23.html


In [104]:
save_problem_report_wikidata_riksdagen(df_problems)


✅ Problem report created: resultsRiksdagenWD/wikidata_riksdagen_problem_report_2026_02_23.html


In [ ]:
 # End timer and calculate duration
end_time = time.time()
elapsed_time = end_time - start_time# Bygg audit-lager för den här etappen

# Print current date and total time
print("Date:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
minutes, seconds = divmod(elapsed_time, 60)
print("Total time elapsed: {:02.0f} minutes {:05.2f} seconds".format(minutes, seconds))
